In [1]:
!pip install gymnasium[toy_tet]

In [2]:
import numpy as np
import random
import gymnasium as gym

env = gym.make("Taxi-v4", render_mode='ansi')

In [3]:
from IPython.display import clear_output
from time import sleep

In [4]:
stat, info = env.reset()
stat

48

In [5]:
print(env.render())

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+




In [6]:
STEPTYPE_FIRST = 0
STEPTYPE_MID = 1
STEPTYPE_LAST = 2

In [7]:
def generate_start_step():
  obs, info = env.reset()
  return {
      'observation' : obs,
      'reward' : 0,
      'step_type' : STEPTYPE_FIRST
  }

In [8]:
def generate_next_step(step, action):
  obs, reward, terminated, truncated, info = env.step(action)
  done = terminated or truncated
  step_type = STEPTYPE_LAST if done else STEPTYPE_MID
  return {
      'observation' : obs,
      'reward' : reward,
      'step_type' : step_type
  }

In [9]:
epsilon = 0.3
Q = np.random.uniform(size = (500, 6))

def get_eps_soft_action(step):
  observ = step['observation']
  n_actions = Q.shape[1]
  action_probs = np.ones(n_actions) * (epsilon / n_actions)
  greedy_action = np.argmax(Q[observ])
  action_probs[greedy_action] += (1 - epsilon)

  return np.random.choice(np.arange(n_actions), p = action_probs)


In [10]:
def get_greedy_action(step):
  observ = step['observation']
  return np.argmax(Q[observ])

In [11]:
def generate_episode(policy_func = get_greedy_action):
  episode = list()
  actions = list()
  frames = list()
  step = generate_start_step()
  episode.append(step)
  frames.append(env.render())

  while step['step_type'] != STEPTYPE_LAST:
    action = policy_func(step)
    step = generate_next_step(step, action)
    episode.append(step)
    actions.append(action)
    frames.append(env.render())

  return episode, actions, frames

In [12]:
gamma = 1.
epsilon = 0.3
Q = np.random.uniform(size = (500, 6))

for _ in range(100000):
  step = generate_start_step()
  action = get_eps_soft_action(step)
  done = False

  while not done:
    next_step = generate_next_step(step, action)

    if next_step['step_type'] == STEPTYPE_LAST:
      state = step['observation']
      idx1 = (state, action)
      Q[idx1] += 0.8 * (next_step['reward'] - Q[idx1])
      done = True

    else:
      next_best_action = get_greedy_action(next_step)
      state = step['observation']
      next_state = next_step['observation']
      idx1 = (state, action)
      idx2 = (next_state, next_best_action)
      Q[idx1] += 0.8 * ((next_step['reward'] + gamma * Q[idx2]) - Q[idx1])
      step = next_step
      action = get_eps_soft_action(next_step)

In [13]:
def print_frames(frames):
  for frame in frames:
    clear_output(wait = True)
    print(frame)
    sleep(0.2)

In [15]:
_, _, frames = generate_episode()
print_frames(frames)

+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (Dropoff)

